- pdate_plan|spawn_agent|exec_command
- function、local_shell、image_generation、web_search、custom。
    - exec_command / write_stdin、update_plan、request_user_input、request_permissions、apply_patch、web_search、view_image、spawn_agent
    -  MCP 和 dynamic tools

### 本地`落盘`

- `RUST_LOG='warn,codex_api=trace,codex_client=trace,codex_core=info,codex_tui=info,codex_rmcp_client=info' codex`

```python
import sqlite3, json
conn = sqlite3.connect('~/.codex/logs_1.sqlite')
msg = conn.execute("select message from logs where id=2673").fetchone()[0]
data = json.loads(msg.split('websocket request: ',1)[1])

# data.keys()
# Out[15]: dict_keys(['type', 'model', 'instructions', 'previous_response_id', 'input', 'tools', 'tool_choice', 'parallel_tool_calls', 'reasoning', 'store', 'stream', 'include', 'prompt_cache_key', 'text', 'client_metadata'])

data['instructions']
data['input'] # history message, <developer> <user> <developer>

print(json.dumps(data['tools'], ensure_ascii=False, indent=2))
```

- `response.create` 请求体
    - `instructions`: 这是一段独立的顶层指令，基本相当于“base/system prompt”。
    - `input`：里面可以是 developer、user、assistant、function_call_output 等，不一定只是“聊天消息”。
    - `previous_response_id`: 这是上下文链接。后续 turn 往往只发增量 input，不是把完整历史重发一遍。

```
{
    "instructions": "...",
    "previous_response_id": "...",
    "input": [...],
    "tools": [...]
}
```

- type & name pair

```
1. function exec_command：执行一个 shell 命令，返回输出或可继续交互的会话 id。
2. function write_stdin：向已有命令会话写入 stdin，并读取最新输出。
3. function update_plan：更新当前任务计划与步骤状态。
4. function request_user_input：向用户发起结构化问题并等待选择/输入。
5. custom apply_patch：用 patch 语法直接编辑文件。
6. web_search：执行网页/图片搜索。
7. function view_image：读取并查看本地图片文件。
8. function spawn_agent：启动一个子代理处理独立子任务。
9. function send_input：给已有子代理继续发送消息或补充上下文。
10. function resume_agent：恢复一个已关闭/暂停的子代理。
11. function wait：等待一个或多个子代理完成。
12. function close_agent：关闭子代理并返回其最后状态。
13. function spawn_agents_on_csv：按 CSV 每行批量启动子代理并汇总结果。
```

- spawn_agent 
    - agent_type: explorer/worker
        - Schema 要求为每个 Worker 分配 “Disjoint write set（不相交的写集合）”
    - fork_context: 如果设为 true，子智能体将继承当前所有的对话记忆。
    - Critical Path Analysis
        - 它要求主模型在调用前进行任务编排：
            - 阻塞性任务 (Blocking Tasks)：如果下一步必须依赖某个结果，主模型应亲自动手（Local execution），避免等待延迟。
            - 旁侧任务 (Sidecar Tasks)：不影响当前进度的独立子任务，应外包给 worker。
    - 什么时候委派(分解)，如何委派（什么时候并行委派），委派之后的处理；
- codex-rs/core/src/tools/spec.rs
    - https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/spec.rs#L1093-L1126
```
Spawn a sub-agent for a well-scoped task. Returns the agent id (and user-facing nickname when available) to use to communicate with this agent. This spawn_agent tool provides you access to smaller but more efficient sub-agents. A mini model can solve many tasks faster than the main model. You should follow the rules and guidelines below to use this tool.

### When to delegate vs. do the subtask yourself
- First, quickly analyze the overall user task and form a succinct high-level plan. Identify which tasks are immediate blockers on the critical path, and which tasks are sidecar tasks that are needed but can run in parallel without blocking the next local step. As part of that plan, explicitly decide what immediate task you should do locally right now. Do this planning step before delegating to agents so you do not hand off the immediate blocking task to a submodel and then waste time waiting on it.
- Use the smaller subagent when a subtask is easy enough for it to handle and can run in parallel with your local work. Prefer delegating concrete, bounded sidecar tasks that materially advance the main task without blocking your immediate next local step.
- Do not delegate urgent blocking work when your immediate next step depends on that result. If the very next action is blocked on that task, the main rollout should usually do it locally to keep the critical path moving.
- Keep work local when the subtask is too difficult to delegate well and when it is tightly coupled, urgent, or likely to block your immediate next step.

### Designing delegated subtasks
- Subtasks must be concrete, well-defined, and self-contained.
- Delegated subtasks must materially advance the main task.
- Do not duplicate work between the main rollout and delegated subtasks.
- Avoid issuing multiple delegate calls on the same unresolved thread unless the new delegated task is genuinely different and necessary.
- Narrow the delegated ask to the concrete output you need next.
- For coding tasks, prefer delegating concrete code-change worker subtasks over read-only explorer analysis when the subagent can make a bounded patch in a clear write scope.
- When delegating coding work, instruct the submodel to edit files directly in its forked workspace and list the file paths it changed in the final answer.
- For code-edit subtasks, decompose work so each delegated task has a disjoint write set.

### After you delegate
- Call wait very sparingly. Only call wait when you need the result immediately for the next critical-path step and you are blocked until it returns.
- Do not redo delegated subagent tasks yourself; focus on integrating results or tackling non-overlapping work.
- While the subagent is running in the background, do meaningful non-overlapping work immediately.
- Do not repeatedly wait by reflex.
- When a delegated coding task returns, quickly review the uploaded changes, then integrate or refine them.

### Parallel delegation patterns
- Run multiple independent information-seeking subtasks in parallel when you have distinct questions that can be answered independently.
- Split implementation into disjoint codebase slices and spawn multiple agents for them in parallel when the write scopes do not overlap.
- Delegate verification only when it can run in parallel with ongoing implementation and is likely to catch a concrete risk before final integration.
- The key is to find opportunities to spawn multiple independent subtasks in parallel within the same round, while ensuring each subtask is well-defined, self-contained, and materially advances the main task.
```

### Responses API

- 旧的是：Chat Completions messages[] => Responses API 的 input[]
    - message 只是 input[] 里的一个 ResponseItem，tool output 是另一个 ResponseItem。

### multi-agents

```sh
• Closed Zeno [explorer] 
• Closed Fermat [explorer]
```
- `explorer`: read-heavy codebase exploration role

### agents in codex

- agent roles 一般定义在本地 ~/.codex/config.toml 或项目内 .codex/config.toml 中；
- Codex ships with built-in roles:
    - default: general-purpose fallback role.
    - worker: execution-focused role for implementation and fixes.
    - explorer: read-heavy codebase exploration role.
    - monitor: long-running command/task monitoring role (optimized for waiting/polling).
- `spwan_agent` is a tool
    - https://github.com/openai/codex/blob/main/codex-rs/core/config.schema.json